In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow.keras.layers as tfl
import tensorflow.keras as tf
from keras.models import Model
import pandas as pd
import h5py as HDF
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2

In [ ]:

train = pd.read_csv("/content/drive/My Drive/Mini_project/insat_3d_ds - Sheet.csv")
train_df, test_df = train_test_split(train, test_size=0.2, random_state=42)
print(train_df)
train_datagen = ImageDataGenerator(rescale=1.0/255.0)
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

      img_name  label
11      34.jpg     34
68      54.jpg     54
129    106.jpg    106
76   58(1).jpg     58
84      61.jpg     61
..         ...    ...
71      56.jpg     56
106  74(2).jpg     74
14   35(1).jpg     35
92   64(1).jpg     64
102     70.jpg     70

[108 rows x 2 columns]


In [ ]:
train_data = train_datagen.flow_from_dataframe(train_df,directory="/content/drive/My Drive/Mini_project/insat3d_ir_cyclone_ds/CYCLONE_DATASET_INFRARED",
                                               x_col="img_name",y_col="label",target_size=(512, 512),batch_size=16,class_mode='raw')
test_data = test_datagen.flow_from_dataframe(test_df,directory="/content/drive/My Drive/Mini_project/insat3d_ir_cyclone_ds/CYCLONE_DATASET_INFRARED",
                                               x_col="img_name",y_col="label",target_size=(512, 512),batch_size=16,class_mode='raw')

Found 108 validated image filenames.
Found 28 validated image filenames.


In [ ]:

#image=cv2.imread("/content/drive/My Drive/Mini_project/insat3d_ir_cyclone_ds/CYCLONE_DATASET_INFRARED/119.jpg")
#image=image/255.0


In [ ]:
def efficient_model():
    base = tf.applications.Xception(weights="imagenet", include_top=False, input_tensor=tfl.Input(shape=(512, 512, 3)))
    base.trainable = True
    flatten= tfl.Dropout(0.4)(base.output)
   # flatten=base.output
    #flatten=tfl.Conv2D(1,1,strides=(1,1))(base_matrix)

    #flatten = base.output
    flatten = tfl.Flatten()(flatten)
    prediction = tfl.Dense(64, activation="relu")(flatten)
    #
    prediction = tfl.Dense(32, activation="relu")(prediction)
    prediction= tfl.Dropout(0.2)(prediction)
    prediction = tfl.Dense(1, activation="linear")(prediction)

    model = Model(inputs=base.input, outputs=prediction)

    return model

In [ ]:
model = efficient_model()
model.compile(optimizer=tf.optimizers.Adam(learning_rate=0.001), loss='mae', metrics=['accuracy'])
save_best = tf.callbacks.ModelCheckpoint("efficient_model.h5", monitor='loss',save_best_only=True, verbose=1)

In [ ]:
model.fit(train_data, epochs=8, callbacks=[save_best])

Epoch 1/8


In [ ]:
train_df
model = tf.models.load_model('./efficient_model.h5')
model.evaluate(test_data)

2/2 [==============================] - 34s 13s/step - loss: 14.6251 - accuracy: 0.0000e+00


[14.625054359436035, 0.0]

In [ ]:
predictions= model.predict(test_data, verbose=1).round(2)

2/2 [==============================] - 44s 19s/step
